# draw market — `mrkt_favourite` segments → model on the most profitable segment

`goals_foragst` space, draw market, on the shared `evaluation.model_check` framework. Two parts:

1. **Raw segment profitability** (descriptive, in-sample) — bet the draw on *every* match in each
   `mrkt_favourite` segment (home / away / balanced), no model. Shows where draws are underpriced.
2. **Model on the best segment** — the SEGMENTER picks, per training fold, the single `mrkt_favourite`
   segment with the best draw edge, and labels only that segment; the model then bets the draw there
   where it beats implied by `buffer`, under the three gates.

This is the **rule-ish** kind of segmenter: instead of clustering, it selects among a handful of
existing categories. Unlike the original notebook (which chose the best segment in-sample, once), the
choice here is made on the TRAIN fold of each split, so it never peeks at the held-out data.

In [1]:
# --- SETUP -------------------------------------------------------------------
# Load the wide ABT for one league group and pull in the model-check framework. Everything that
# never changes between checks (folds, ROI scoring, the honest per-fold engine, the three gates,
# the registry write) lives in `evaluation.model_check`; this notebook only defines the parts that
# DO change — the space, the target, the segmenter, and the model.
GROUP = "major"   # league group: which <group>/ ABT to validate
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline

from evaluation.model_check import (
    run_check, evaluate, collect_bets, pooled_roi, roi_grouped,
    season_folds, expanding_folds,
)

abt = pd.read_parquet(f"C:/Users/twluc/frame/01_data/03_abt/{GROUP}/abt.parquet")

In [2]:
# --- SPACE (modifiable) ------------------------------------------------------
# The bet-signal model sees the four running goals for/against features (the goals_foragst space).
FEATURES = [
    "hmt_season_goals_for_avg", "hmt_season_goals_agst_avg",
    "awt_season_goals_for_avg", "awt_season_goals_agst_avg",
]

In [3]:
# --- TARGET (modifiable) -----------------------------------------------------
OUTCOME = "t_draw_flg"              # 1 when the match ended in a draw
IMPLIED = "mrkt_draw_impl"          # implied draw prob we bet against = 1 / mrkt draw odds
SEGMENT_COL = "mrkt_favourite"      # the category we segment on: home / away / balanced

In [4]:
# --- ABT FILTERING -----------------------------------------------------------
MIN_GAME_NUMBER = 8
matches = abt[
    (abt["hmt_game_number"] > MIN_GAME_NUMBER)
    & (abt["awt_game_number"] > MIN_GAME_NUMBER)
].dropna(subset=FEATURES + [SEGMENT_COL, IMPLIED, OUTCOME, "league", "season"]).copy()

print(f"matches: {len(matches)}   seasons: {sorted(matches['season'].unique())}")
print(f"segments ({SEGMENT_COL}): {matches[SEGMENT_COL].value_counts().to_dict()}")

matches: 5533   seasons: ['2223', '2324', '2425', '2526']
segments (mrkt_favourite): {'balanced': 2767, 'home': 1974, 'away': 792}


In [5]:
# --- SEGMENTER (modifiable) — rule/column segmenter on mrkt_favourite --------
# No clustering: segment by an existing categorical column (mrkt_favourite: home / away / balanced) and
# use the SAME keep_segment as the kmeans notebooks to choose which categories to bet.
#   keep="best" -> the single highest-edge category (even if negative);  keep="profitable" -> all positive.
def keep_segment(train, labels, keep="profitable", min_matches=100):
    frame = pd.DataFrame({"label": labels.to_numpy(),
                          "won": train[OUTCOME].to_numpy(),
                          "implied": train[IMPLIED].to_numpy()})
    stats = frame.groupby("label").agg(won=("won", "mean"), implied=("implied", "mean"), n=("won", "size"))
    edge = (stats["won"] - stats["implied"])[stats["n"] >= min_matches]
    if keep == "best":
        return {edge.idxmax()} if len(edge) else set()   # single highest-edge segment (sign ignored)
    return set(edge[edge > 0].index)                     # all positive-edge segments

def column_segmenter(column, keep="best", min_matches=50):
    def segmenter(train):
        kept = keep_segment(train, train[column], keep=keep, min_matches=min_matches)
        def labeler(rows):
            return rows[column].where(rows[column].isin(kept))   # non-kept categories -> NaN -> not bet
        return labeler
    return segmenter

segmenter = column_segmenter(SEGMENT_COL, keep="best")   # keep="profitable" -> all positive-edge categories

In [6]:
# --- MODEL (modifiable) ------------------------------------------------------
# The bet-signal model: given the space features, predict the probability of the target outcome.
# Today it is polynomial-expanded features (squares + pairwise products), standardised, then logistic
# regression. To try a different method (random forest, XGBoost, ...) swap this one factory — the
# only contract the framework needs is an sklearn-style estimator with .fit(X, y) and .predict_proba(X).
def make_model():
    return make_pipeline(PolynomialFeatures(degree=2, include_bias=False),
                         StandardScaler(), LogisticRegression(max_iter=3000))

BUFFER = 0.02   # only bet when the model probability beats the implied one by at least this margin

## Part 1 — raw draw profitability per segment (descriptive, in-sample)
Bet the draw on every match in each segment (no model). ROI under implied-probability staking, mrkt
odds. Just a map of where draws look underpriced — not evidence, and not what the gates below judge.

In [7]:
print("raw draw ROI per mrkt_favourite segment (bet draw on all matches, mrkt):")
for seg in sorted(matches[SEGMENT_COL].unique()):
    sub = matches[matches[SEGMENT_COL] == seg]
    roi = pooled_roi(sub[IMPLIED], sub[OUTCOME].astype(int))
    frame = pd.DataFrame({"implied": sub[IMPLIED], "won": sub[OUTCOME].astype(int), "fold": sub["season"]})
    per_season = roi_grouped(frame, "fold")
    print(f"  {seg:9s} ROI {roi:+.4f}  ({len(sub)} matches)  "
          + "  ".join(f"{s}:{v:+.3f}" for s, v in per_season.items()))

raw draw ROI per mrkt_favourite segment (bet draw on all matches, mrkt):
  away      ROI -0.0602  (792 matches)  2223:-0.048  2324:-0.095  2425:-0.014  2526:-0.088
  balanced  ROI -0.0392  (2767 matches)  2223:-0.089  2324:+0.062  2425:-0.070  2526:-0.055
  home      ROI -0.0282  (1974 matches)  2223:-0.090  2324:+0.085  2425:-0.079  2526:-0.022


## Summary — model on the train-best segment, three gates
The segmenter picks the best `mrkt_favourite` segment per training fold; the model bets the draw
there. Row logged to `model_checks/results.md`.

In [8]:
_ = run_check(matches, features=FEATURES, outcome=OUTCOME, implied=IMPLIED,
              segmenter=segmenter, make_model=make_model, buffer=BUFFER,
              source="goals_foragst_draw_favourite_best_lr", space="goals_foragst", target="draw",
              abt_filter="major, gn>8, dropna", segmentation="mrkt_favourite",
              segment_filter="best train-edge segment")

goals_foragst_draw_favourite_best_lr   pooled -0.090 (514 bets)   [mrkt]

GATE 1  LOSO — ROI by test season           FAIL  (+1/4 seasons)
    2223 -0.256    2324 -0.131    2425 -0.046    2526 +0.092

GATE 2  league — ROI by league / drop best   FAIL
    england +0.007    italy -0.013    germany -0.023    spain -0.096    france -0.371
    pooled without best league (england): -0.131

GATE 3  walk-forward — ROI by test season    FAIL
    2425 -0.128    2526 +0.092
